In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

from langchain_ollama import ChatOllama

import pandas as pd
import torch

c:\Users\valen\anaconda3\envs\torch_gpu\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\valen\anaconda3\envs\torch_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

c:\Users\valen\anaconda3\envs\torch_gpu\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\valen\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3966.15it/s]
BertModel LOAD REPORT

In [3]:
df=pd.read_csv('document.csv')
texts=df['content'].tolist()
documents=[Document(page_content=text) for text in texts]
vectorstore=FAISS.from_documents(documents,embedding_model)
retriever=vectorstore.as_retriever()

In [4]:
documents

[Document(metadata={}, page_content='All students must wear lab coats and safety goggles during laboratory sessions.'),
 Document(metadata={}, page_content='Food and drinks are strictly prohibited inside the laboratory.'),
 Document(metadata={}, page_content='Chemical spills must be reported immediately to the laboratory supervisor.'),
 Document(metadata={}, page_content='Students must complete safety training before using advanced laboratory equipment.'),
 Document(metadata={}, page_content='Laboratory waste must be disposed of according to hazardous material guidelines.')]

In [7]:
def format_docs(documents):
    return '\n\n'.join(page.page_content for page in documents)

rag_prompt=PromptTemplate.from_template(
    """
    Use the context below to answer the Question.
    If you don't know, say that you don't know

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

chat_llm=ChatOllama(
    model='qwen2.5',
    reasoning=False,
    timeout=None
)

rag_chain=(
    {'context':retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | chat_llm
    | StrOutputParser()
)

In [8]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig
from datasets import Dataset

C:\Users\valen\AppData\Local\Temp\ipykernel_3752\681498119.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\valen\AppData\Local\Temp\ipykernel_3752\681498119.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\valen\AppData\Local\Temp\ipykernel_3752\681498119.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

In [9]:
df_evaluation=pd.read_csv('eval.csv')
rag_question=df_evaluation['question'].tolist()
ground_truth=df_evaluation['ground_truth'].tolist()

evaluation_dict={
    'question':[],
    'answer':[],
    'retrieved_contexts':[],
    'ground_truth':[]
}

In [10]:
for i, question in enumerate(rag_question):
    try:
        retrieved_docs=retriever.invoke(question)
        context=[page.page_content for page in retrieved_docs]
        context_formatted=format_docs(retrieved_docs)
        prompt=rag_prompt.format(question=question,context=context_formatted)
        response=chat_llm.invoke(prompt).content
        
        evaluation_dict['question'].append(question)
        evaluation_dict['answer'].append(response)
        evaluation_dict['retrieved_contexts'].append(context)
        evaluation_dict['ground_truth'].append(ground_truth[i])
    except Exception as e:
        print(f'Error at index-{i} | {e}')


In [ ]:
import os
os.environ["RAGAS_DEBUG"] = "true"

In [11]:
evaluation_dataset=Dataset.from_dict(evaluation_dict)

result=evaluate(
    evaluation_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=chat_llm,
    embeddings=embedding_model,
    raise_exceptions=False,
    run_config=RunConfig(timeout=600, max_workers=1)
)

Evaluating: 100%|██████████| 20/20 [12:47<00:00, 38.39s/it]


In [12]:
result=result.to_pandas()
result

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What protective equipment must students wear i...,[Students must complete safety training before...,Students must wear lab coats and safety goggle...,Lab coats and safety goggles,1.0,NaN,0.5,1.0
1,Are food and drinks allowed inside the laborat...,[Food and drinks are strictly prohibited insid...,"No, food and drinks are strictly prohibited in...",No,1.0,NaN,1.0,1.0
2,What should students do if a chemical spill oc...,[Chemical spills must be reported immediately ...,Students should report the chemical spill imme...,Report it immediately to the laboratory superv...,1.0,NaN,1.0,1.0
3,What is required before using advanced laborat...,[Students must complete safety training before...,"Before using advanced laboratory equipment, st...",Complete safety training,1.0,NaN,1.0,1.0
4,How should laboratory waste be handled?,[Laboratory waste must be disposed of accordin...,"According to the context provided, laboratory ...",Disposed of according to hazardous material gu...,1.0,NaN,1.0,1.0


In [13]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent

In [14]:
@tool
def lab_rules_tool(query: str)->str:
    """
    Answer question about laboratory regulation
    """
    response=rag_chain.invoke(query)
    return response

@tool
def calculator_tool(query: str)->str:
    """
    Perform general mathematic calculations.
    """

    try:
        return str(eval(query,{'__builtin__':None},{}))
    except Exception as e:
        return e
    
@tool
def percentage_tool(query: str)->str:
    """
    Calculate percentage
    Input: value, total
    Example: 45/90
    """
    try:
        value, total=map(float,query.split(','))
        return (f'Percentage: {value/total:.2f}')
    except Exception as e:
        return e
    
tools=[
    lab_rules_tool,
    calculator_tool,
    percentage_tool
]


In [15]:
system_prompt="""
    you are LabHouseAI, an intelligent laboratory academic assistant

    use lab_rules_tool to answer questions about laboratory regulations
    use calculator_tool to calculate mathematical expression, formula, etc
    use percentage_tool to calculate percentage

    always use the appropiate tool based on the user's request
    answer clearly and accurately
"""

agent=create_agent(
    model=chat_llm,
    tools=tools,
    system_prompt=system_prompt
)

In [18]:
while True:
    query=input("enter a prompt or type 'exit' or 'quit' to quit")
    if query.lower() in ['exit','quit']:
        print('quitting...')
        break
    try:
        response=agent.invoke({'messages':[HumanMessage(content=query)]})
        answer=response['messages'][-1].content
        print(f'Prompt: {query}')
        print(f'Answer: {answer}')
    except Exception as e:
        print(e)

Prompt: what is 5*123?
Answer: The result of 5 multiplied by 123 is 615.
Prompt: thank you
Answer: You're welcome! If you have any questions or need assistance, feel free to ask.
Prompt: what are you?
Answer: I am LabHouseAI, an intelligent laboratory academic assistant designed to help with various tasks related to laboratory regulations, calculations, and more. How can I assist you today?
quitting...
